In [0]:

SILVER_TABLE = "retail_de_27.silver.orders"

silver_df = spark.table(SILVER_TABLE)

display(silver_df)
silver_df.printSchema()


In [0]:

from pyspark.sql import functions as F

daily_sales_df = (
    silver_df
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum(
            F.col("quantity") * F.col("price")
        ).alias("total_revenue")
    )
    .orderBy("order_date")
)

display(daily_sales_df)


In [0]:

CATALOG = "retail_de_27"

# Create Gold schema
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold"
)

# Save Daily Sales
GOLD_TABLE = f"{CATALOG}.gold.daily_sales"

(
    daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Gold table created successfully")


In [0]:

gold_df = spark.table(GOLD_TABLE)

display(gold_df)

print("Total records:", gold_df.count())
